In [1]:
import sys
import polars as pl
import plotly.express as px
sys.path.insert(0, '..')
from fs_thesis import sql, show
from sklearn.metrics import classification_report, confusion_matrix
import plotly.figure_factory as ff
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

# 1. Data Pipeline Overview
This notebook uses the centralized data pipeline defined in `fs_thesis.data_loader`. Below is a documentation of the logic encapsulated in `load_final_data()`.

### A. Patient Demographics (Baseline)
We extract the following baseline features from the first admission (`t0_time`):
- **Identifier**: `subject_id`
- **Demographics**: `gender`, `anchor_age`, `race`, `marital_status`, `language`, `insurance`
- **Context**: `admission_type`
- **BMI**: Median BMI from `hosp.omr` (Left Join, missing values are handled by TabPFN)

### B. Event Definition (Target)
The event is defined as the **first occurrence** of a specific ICD diagnosis (e.g., Heart Failure `I50%`).
- **Event Time**: Timestamp of the diagnosis.
- **Censoring**: If no event occurs, the patient is censored at the date of death (`dod`) or end of follow-up.

### C. Target Calculation Logic
The target variable is derived based on the time-to-event (`duration`):
1. **Calculate Duration**:
   - `t_event` = Days from baseline to diagnosis.
   - `t_death` = Days from baseline to death.
   - Priority: Event Time > Death Time > Fallback (2000 days).
   - Negative durations are clipped to 0.

2. **Define Classes (`target`)**:
   - **Class 0 (Early Event)**: Event occurs $\le$ 365 days.
   - **Class 1 (Late Event)**: Event occurs $>$ 365 days.
   - **Class 2 (Censored/Control)**: No event observed (censored or healthy).

In [2]:
from fs_thesis.data_loader import load_final_data
df_final = load_final_data()

In [3]:
# Prüfen wie viele Missings wir haben
print("Missing BMI from Join, before feature engineering: TabPFN will handle these missing values, but it's good to know how many we have.")
print(f"Missing BMI: {df_final['bmi'].null_count()} of {len(df_final)}")

Missing BMI from Join, before feature engineering: TabPFN will handle these missing values, but it's good to know how many we have.
Missing BMI: 98306 of 223452


# 2. Preprocessing
## Splitting

In [4]:
from fs_thesis.preprocessing import preprocess_data, balance_data, get_X_y
df_train, df_val, df_test = preprocess_data(df_final)

Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)


## Sampling (Balancing)

In [5]:
# Balance (only for train data!)
df_balanced = balance_data(df_train, n_samples=3000)

# split Features & Target (all Sets!)
X_train, y_train = get_X_y(df_balanced)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

# Jetzt passt auch der fucking Print
print(f"Train (balanced): {len(y_train)} | Val (real): {len(y_val)} | Test (real): {len(y_test)}")

Train (balanced): 9000 | Val (real): 35753 | Test (real): 44691


# 3. Training
## Classifier

In [6]:
from tabpfn import TabPFNClassifier
classifier = TabPFNClassifier(device='mps') # Zurück auf CPU, für schnellere Vorhersagen bei kleinen N

## Fit

In [7]:
print("Start fitting...")
classifier.fit(X_train, y_train)
print("Training done!")

Start fitting...
Training done!


## Predict

In [8]:
# 3. Vorhersage (Validierung)
# Achtung: TabPFN ist bei Prediction auf vielen Daten (Validation Set = groß) langsam!
# Wir nehmen erstmal 1000 Samples für den schnellen Check.
X_val_sample = X_val.iloc[:500]
y_val_sample = y_val[:500]

# Sample run (faster)
sample = True
# For importance analysis
skip = False

print("Start predict (3min)...")
y_val_pred = classifier.predict(X_val_sample) if sample else classifier.predict(X_val)
print("Done!")

Start predict (3min)...
Done!


# 4. Validation (val_set for optimizing)

In [9]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("Evaluating (3min)...")
# Alle Classen (shape: n_samples x 3)
y_proba = classifier.predict_proba(X_val_sample) if sample else classifier.predict_proba(X_val)
# multi_class='ovr' berechnet den Durchschnitt der AUCs aller Klassen
roc_auc = roc_auc_score(y_val_sample if sample else y_val, y_proba, multi_class='ovr', average='macro')

Evaluating (3min)...


In [10]:
# custom_thresholds = [0.6, 0.6, 0.2] # Beispiel: Höhere Hürde für Class 0 & 1

# def custom_predict(probas, thresholds):
#     # Gewichtet die Wahrscheinlichkeiten mit den Thresholds
#     adjusted_probas = probas / np.array(thresholds)
#     return np.argmax(adjusted_probas, axis=1)

# y_pred_adjusted = custom_predict(y_proba, custom_thresholds)

In [11]:
# 3. Vergleich der Ergebnisse
# print("--- Ursprünglicher F1-Macro ---")
# print(f1_score(y_val, y_val_pred, average='macro'))

# print("\n--- Optimierter F1-Macro ---")
# print(f1_score(y_val, y_pred_adjusted, average='macro'))

In [12]:
if sample:
    print(f"Accuracy: {accuracy_score(y_val_sample, y_val_pred):.2f}")
    print(f"AUC - ROC Score: {roc_auc:.2f}")
    print("\nClassification Report:")
    print(f"{classification_report(y_val_sample, y_val_pred, 
                                   target_names=['early (<1J)', 'late (>1J)', 'healthy'])}")
else:
    print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.2f}")
    print(f"AUC - ROC Score: {roc_auc:.2f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_val_pred, 
                                target_names=['early (<1J)', 'late (>1J)', 'healthy']))

# print("--- Mit angepassten Thresholds (Sample) ---")
# if sample:
#     print(f"Accuracy: {accuracy_score(y_val_sample, y_pred_adjusted):.2f}")
#     print(f"AUC - ROC Score: {roc_auc:.2f}")
#     print("\nClassification Report:")
#     print(f"{classification_report(y_val_sample, y_pred_adjusted, 
#                                    target_names=['early (<1J)', 'late (>1J)', 'healthy'])}")
# else:
#     print(f"Accuracy: {accuracy_score(y_val, y_pred_adjusted):.2f}")
#     print(f"AUC - ROC Score: {roc_auc:.2f}")
#     print("\nClassification Report:")
#     print(classification_report(y_val, y_pred_adjusted, 
#                                 target_names=['early (<1J)', 'late (>1J)', 'healthy']))

Accuracy: 0.54
AUC - ROC Score: 0.77

Classification Report:
              precision    recall  f1-score   support

 early (<1J)       0.10      0.57      0.17        21
  late (>1J)       0.09      0.80      0.16        15
     healthy       0.99      0.53      0.69       464

    accuracy                           0.54       500
   macro avg       0.39      0.63      0.34       500
weighted avg       0.92      0.54      0.65       500



# 4.1 Visualization Data

In [13]:
# Check: Zusammenhang zwischen Alter und Versicherung (Medicare)
# Wir schauen, wie hoch das Durchschnittsalter pro Versicherungsgruppe ist.
import plotly.express as px

if sample: 
    X_val = X_val_sample
    y_val = y_val_sample
else: 
    X_val = X_val
    y_val = y_val

df_analyze = X_val.copy() # Kopie für Analyse
if hasattr(df_analyze, "to_pandas"):
    df_analyze = df_analyze.to_pandas()

# Boxplot zeigt klar: Medicare-Patienten sind fast alle 65+
fig = px.box(df_analyze, x="insurance", y="anchor_age", 
             title="'Medicare'-Effekt, be causion with age for TabPFN",
             points="all", 
             color="insurance")
fig.show()

In [14]:
risk_score = y_proba[:, 0]

df_analyze['risk_score'] = risk_score
df_analyze['true_label'] = y_val

### BMI Analyse

In [15]:
bins = [0, 18.5, 25, 30, 100]
labels = ['Untergewicht (<18.5)', 'Normal (18.5-25)', 'Übergewicht (25-30)', 'Adipositas (>30)']
df_analyze['bmi_group'] = pd.cut(df_analyze['bmi'], bins=bins, labels=labels)

df_bmi_mean = df_analyze.groupby('bmi_group', observed=True)['risk_score'].mean().reset_index()

fig1 = px.bar(
    df_bmi_mean, 
    x='bmi_group', 
    y='risk_score',
    text_auto='.1%', # Zeigt %-Wert direkt auf dem Balken
    title='Avg risk by BMI-Group',
    labels={'risk_score': 'Risk Score', 'bmi_group': 'BMI Group'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig1.update_layout(yaxis_tickformat='.0%') # Y-Achse als Prozent formatieren
fig1.show()

### Age Analyse

In [16]:
df_analyze['age_group'] = pd.cut(
    df_analyze['anchor_age'], 
    bins=[0, 20, 30, 40, 50, 60, 70, 80, 90, 120],  # 0-20, 20-30, ..., 90-120
    labels=['<20', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90+']
)
df_analyze['age_group'] = df_analyze['age_group'].astype(str)

df_age_mean = df_analyze.groupby('age_group')['risk_score'].mean().reset_index()

fig2 = px.bar(
    df_age_mean, 
    x='age_group', 
    y='risk_score',
    text_auto='.1%',
    title='Risk by Age Group',
    labels={'risk_score': 'Risk Score', 'age_group': 'Age Group'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig2.update_layout(yaxis_tickformat='.0%')
fig2.show()

### Social - Economy

In [17]:
df_ins_mean = df_analyze.groupby(['insurance', 'gender'])['risk_score'].mean().reset_index()

fig3 = px.bar(
    df_ins_mean, 
    x='insurance', 
    y='risk_score', 
    color='gender', 
    barmode='group',
    text_auto='.1%',
    title='Risk by Gender and Insurance',
    labels={'risk_score': 'Risk Score', 'insurance': 'Insurance'},
    template="plotly_white"
)
fig3.update_layout(yaxis_tickformat='.0%')
fig3.show()

# 5. Visualisation Prediction Performance

In [18]:


if sample: 
    X_val = X_val_sample
    y_val = y_val_sample
else: 
    X_val = X_val
    y_val = y_val
# 1. Normalisierung der Confusion Matrix (Zeilenweise)
# Wie viel % der tatsächlichen Klasse wurden wie vorhergesagt?
# WICHTIG: Wir vergleichen hier mit y_val_sample, da wir nur darauf vorhergesagt haben!

cm = confusion_matrix(y_val, y_val_pred)
cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

labels = ['early (<1J)', 'late (1-3J)', 'healthy']

# 2. Text für die Boxen (Absolute Zahl + Prozent)
annot_text = [
    [f"<b>{val}</b><br>({perc:.1%})" for val, perc in zip(row_val, row_perc)]
    for row_val, row_perc in zip(cm, cm_perc)
]

# 3. Interaktive Heatmap
fig = ff.create_annotated_heatmap(
    cm_perc, 
    x=labels, 
    y=labels, 
    annotation_text=annot_text, 
    colorscale='Reds' # 'Reds' hebt die Treffer besser hervor
)
if sample:
    fig.update_layout(
        title='Validation Check: Confusion Matrix (1000 Samples)',
        xaxis_title="Vorhersage des Modells",
        yaxis_title="Tatsächlicher Verlauf (MIMIC-Daten)",
        template="plotly_white",
        height=600
    )
else:
    fig.update_layout(
        title='Validation Check: Confusion Matrix (3000 Samples)',
        xaxis_title="Vorhersage des Modells",
        yaxis_title="Tatsächlicher Verlauf (MIMIC-Daten)",
        template="plotly_white",
        height=600
    )

fig.show()

In [19]:
import plotly.graph_objects as go

# Daten aus deiner Confusion Matrix extrahieren
# Reihenfolge: [Früh, Spät, Gesund]

if sample: 
    X_val = X_val_sample
    y_val = y_val_sample
else: 
    X_val = X_val
    y_val = y_val

cm = confusion_matrix(y_val, y_val_pred)

# Labels für die Knoten
label_list = [
    "real: early", "real: late", "real: healthy", # Quellen (Links)
    "predicted: early", "predicted: late", "predicted: healthy" # Ziele (Rechts)
]

# Definition der Flüsse (Sankey-Struktur)
source = [0, 0, 0, 1, 1, 1, 2, 2, 2] # Index der Quellen
target = [3, 4, 5, 3, 4, 5, 3, 4, 5] # Index der Ziele
value = cm.flatten() # Die Zahlen aus deiner Matrix

# Farben definieren (Grün für korrekt, Rot für Fehler)
color_link = [
    'rgba(255, 90, 90, 0.4)', 'rgba(255, 90, 90, 0.2)', 'rgba(255, 90, 90, 0.1)', # Von Früh
    'rgba(255, 127, 14, 0.2)', 'rgba(255, 127, 14, 0.4)', 'rgba(255, 127, 14, 0.1)', # Von Spät
    'rgba(44, 160, 44, 0.1)', 'rgba(44, 160, 44, 0.1)', 'rgba(44, 160, 44, 0.4)'    # Von Gesund
]

fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15, thickness = 20, line = dict(color = "black", width = 0.5),
      label = label_list, color = "grey"
    ),
    link = dict(
      source = source, target = target, value = value, color = color_link
  ))])

fig.update_layout(title_text="Patient-Flow: Reality vs. Predicted", font_size=12)
fig.show()

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
import plotly.express as px

if skip:
    print("skipped, dauert zu lang der Scheiss")
else:
    print("Berechne Feature Importance (n_repeats=3, features=8, sample=500) = 3 * 8 * Dauer Predict bei 500 samples...")

    if sample: 
        X_val = X_val_sample
        y_val = y_val_sample
    else:    
        X_val = X_val
        y_val = y_val

    result = permutation_importance(
        classifier, 
        X_val, y_val, 
        n_repeats=3, 
        random_state=42, 
        n_jobs=1
    )

    # Ergebnisse in ein DataFrame gießen
    importance_df = pd.DataFrame({
        'Feature': ['gender', 'anchor_age', 'insurance', 'language', 'marital_status', 'race', 'admission_type', 'bmi'],
        'Importance': result.importances_mean,
        'Std_Dev': result.importances_std
    }).sort_values(by='Importance', ascending=True) # Aufsteigend für horizontalen Plot

    # Plotly Bar Chart
    fig = px.bar(
        importance_df, 
        x='Importance', 
        y='Feature', 
        orientation='h',
        title='Which social facts are important?',
        labels={'Importance': '(Mean Decrease Accuracy)'}, # Wissenschaftlicher Label
        error_x='Std_Dev', # Zeigt die Variabilität der Wichtigkeit
        template="plotly_white",
        color='Importance',
        color_continuous_scale='Reds'
    )

    # NEU: Prozent-Formatierung für bessere Lesbarkeit
    fig.update_layout(height=500, xaxis_tickformat='.1%')
    fig.show()

Berechne Feature Importance (n_repeats=3, features=8,sample=500) = 3 * 8 * Dauer Predict bei 500 samples...


# 6 Robustness Check

In [ ]:
import time
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from sklearn.metrics import recall_score, precision_score
import warnings

# Warnungen unterdrücken
warnings.filterwarnings('ignore', message='.*Running on CPU with more than 200 samples.*')


# --- Configuration ---
N_LOOPS = 30
RESULTS_DIR = Path("/Users/andrey/Repositories/fs-thesis")
RESULTS_DIR.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print(f"Starting Robustness Check ({N_LOOPS} runs)...")
print(f"Device: CPU (Warnings suppressed)")

results = []

configs = [
    {"name": "small", "N_ensemble": 32, "n_samples": 150},      # Schnell, konservativ
    {"name": "medium", "N_ensemble": 32, "n_samples": 300},     # aktueller Standard
    {"name": "large", "N_ensemble": 32, "n_samples": 340},      # Max ohne Limit (340*3≈1020)
]

for config in configs:
    print(f"\n{'='*60}")
    print(f"Config: {config['name']} | N_ensemble={config['N_ensemble']} | n_samples={config['n_samples']}")
    print(f"{'='*60}")
    
    # Loop über verschiedene Balancing-Seeds
    for i in tqdm(range(N_LOOPS), desc=f"Training {config['name']}"):
        
        # 1. Classifier initialisieren
        try:
            classifier = TabPFNClassifier(device='cpu', n_estimators=config['N_ensemble'])
        except TypeError:
            # Fallback: Falls n_estimators nicht existiert
            print(f"⚠️  n_estimators wird nicht unterstützt, verwende Standard-Config")
            classifier = TabPFNClassifier(device='cpu')
        
        # 2. Daten neu samplen
        df_balanced_loop = balance_data(df_train, n_samples=config['n_samples'], seed=42 + i)
        X_train_loop, y_train_loop = get_X_y(df_balanced_loop)
        
        # Warnung bei TabPFN-Limit
        if len(X_train_loop) > 1024:
            print(f"\n⚠️  WARNUNG (Run {i}): Training-Set ({len(X_train_loop)}) > TabPFN-Limit (1024)")
        
        # 3. Modell fitten
        classifier.fit(X_train_loop, y_train_loop)
        
        # 4. Validieren
        y_pred_loop = classifier.predict(X_val)
        y_proba_loop = classifier.predict_proba(X_val)
        
        # 5. Metriken berechnen
        f1_per_class = f1_score(y_val, y_pred_loop, average=None)
        
        results.append({
            'run_id': i,
            'config_name': config['name'],       
            'n_ensemble': config['N_ensemble'], 
            'n_samples': config['n_samples'],
            'accuracy': accuracy_score(y_val, y_pred_loop),
            'roc_auc_macro': roc_auc_score(y_val, y_proba_loop, multi_class='ovr', average='macro'),
            'f1_macro': f1_score(y_val, y_pred_loop, average='macro'),
            'recall_macro': recall_score(y_val, y_pred_loop, average='macro'),
            'precision_macro': precision_score(y_val, y_pred_loop, average='macro'),
            # Klassen-spezifisch
            'f1_class_0_early': f1_per_class[0],
            'f1_class_1_late': f1_per_class[1],
            'f1_class_2_healthy': f1_per_class[2],
            'seed': 42 + i
        })

# 6. Ergebnisse speichern und auswerten
if not results:
    print("\n❌ FEHLER: Die Results-Liste ist leer!")
else:
    df_results = pd.DataFrame(results)
    save_file = RESULTS_DIR / f"robustness_metrics_{timestamp}.csv"
    
    try:
        df_results.to_csv(save_file, index=False)
        print(f"\n✅ Datei gespeichert: {save_file}")
        
        # Zusammenfassung pro Config
        print(f"\n{'='*60}")
        print("ZUSAMMENFASSUNG")
        print(f"{'='*60}")
        
        for config_name in df_results['config_name'].unique():
            subset = df_results[df_results['config_name'] == config_name]
            print(f"\n📊 Config: {config_name}")
            print(f"   F1 (Macro):     {subset['f1_macro'].mean():.4f} ± {subset['f1_macro'].std():.4f}")
            print(f"   Accuracy:       {subset['accuracy'].mean():.4f} ± {subset['accuracy'].std():.4f}")
            print(f"   ROC-AUC (Macro): {subset['roc_auc_macro'].mean():.4f} ± {subset['roc_auc_macro'].std():.4f}")
            print(f"   F1 Early:       {subset['f1_class_0_early'].mean():.4f} ± {subset['f1_class_0_early'].std():.4f}")
            print(f"   F1 Late:        {subset['f1_class_1_late'].mean():.4f} ± {subset['f1_class_1_late'].std():.4f}")
            print(f"   F1 Healthy:     {subset['f1_class_2_healthy'].mean():.4f} ± {subset['f1_class_2_healthy'].std():.4f}")
        
    except Exception as e:
        print(f"\n❌ Fehler beim Speichern: {e}")

Starting Robustness Check (30 runs)...
Device: CPU (Warnings suppressed)

Config: small | N_ensemble=32 | n_samples=150


Training small:   0%|          | 0/30 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Claude Code:

In [ ]:
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score, roc_auc_score
import time

# Warnungen unterdrücken
warnings.filterwarnings('ignore', message='.*Running on CPU with more than 200 samples.*')

# --- Configuration für Overnight Run ---
N_LOOPS = 100  # 🔥 Deutlich mehr Loops für robuste Statistik
RESULTS_DIR = Path("/Users/andrey/Repositories/fs-thesis")
RESULTS_DIR.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Logging für Overnight-Run
log_file = RESULTS_DIR / f"overnight_log_{timestamp}.txt"

def log_message(msg):
    """Schreibt in Logfile und printet"""
    print(msg)
    with open(log_file, 'a') as f:
        f.write(f"{datetime.now().strftime('%H:%M:%S')} | {msg}\n")

log_message("="*80)
log_message(f"OVERNIGHT ROBUSTNESS RUN GESTARTET")
log_message(f"Timestamp: {timestamp}")
log_message(f"Loops: {N_LOOPS}")
log_message("="*80)

results = []
start_time_total = time.time()

# 🔥 ERWEITERTE CONFIGS FÜR UMFASSENDE ANALYSE
configs = [
    # Sample Size Variation (bei konstantem Ensemble)
    {"name": "small_samples", "N_ensemble": 32, "n_samples": 150},
    {"name": "medium_samples", "N_ensemble": 32, "n_samples": 300},
    {"name": "large_samples", "N_ensemble": 32, "n_samples": 340},  # Max ohne Limit
    
    # Ensemble Size Variation (bei konstanten Samples)
    {"name": "low_ensemble", "N_ensemble": 16, "n_samples": 300},
    {"name": "standard_ensemble", "N_ensemble": 32, "n_samples": 300},
    {"name": "high_ensemble", "N_ensemble": 64, "n_samples": 300},
    
    # Kombinationen für "Best Model"
    {"name": "best_balanced", "N_ensemble": 64, "n_samples": 340},
    {"name": "ultra_ensemble", "N_ensemble": 128, "n_samples": 300},  # Wenn du Zeit hast
]

log_message(f"\nAnzahl Configs: {len(configs)}")
log_message(f"Geschätzte Gesamtzeit: {len(configs) * N_LOOPS * 15 / 60:.1f} Minuten")
log_message(f"(bei ~15 Sekunden pro Loop)\n")

# Checkpoint-Funktion (speichert Zwischenergebnisse)
def save_checkpoint(results, config_idx):
    """Speichert Zwischenergebnisse nach jeder Config"""
    if results:
        df_checkpoint = pd.DataFrame(results)
        checkpoint_file = RESULTS_DIR / f"checkpoint_{timestamp}_config{config_idx}.csv"
        df_checkpoint.to_csv(checkpoint_file, index=False)
        log_message(f"   💾 Checkpoint gespeichert: {checkpoint_file.name}")

# Hauptloop
for config_idx, config in enumerate(configs, 1):
    config_start_time = time.time()
    
    log_message(f"\n{'='*80}")
    log_message(f"[{config_idx}/{len(configs)}] Config: {config['name']}")
    log_message(f"  N_ensemble={config['N_ensemble']}, n_samples={config['n_samples']}")
    log_message(f"{'='*80}")
    
    config_results = []
    
    # Loop über verschiedene Seeds
    for i in tqdm(range(N_LOOPS), desc=f"[{config_idx}/{len(configs)}] {config['name']}"):
        
        try:
            # 1. Classifier initialisieren
            try:
                classifier = TabPFNClassifier(device='cpu', n_estimators=config['N_ensemble'])
            except TypeError:
                classifier = TabPFNClassifier(device='cpu')
            
            # 2. Daten neu samplen
            df_balanced_loop = balance_data(df_train, n_samples=config['n_samples'], seed=42 + i)
            X_train_loop, y_train_loop = get_X_y(df_balanced_loop)
            
            # 3. Modell fitten
            classifier.fit(X_train_loop, y_train_loop)
            
            # 4. Validieren
            y_pred_loop = classifier.predict(X_val)
            y_proba_loop = classifier.predict_proba(X_val)
            
            # 5. Metriken berechnen
            f1_per_class = f1_score(y_val, y_pred_loop, average=None)
            
            result = {
                'run_id': i,
                'config_name': config['name'],       
                'n_ensemble': config['N_ensemble'], 
                'n_samples': config['n_samples'],
                'accuracy': accuracy_score(y_val, y_pred_loop),
                'roc_auc_macro': roc_auc_score(y_val, y_proba_loop, multi_class='ovr', average='macro'),
                'f1_macro': f1_score(y_val, y_pred_loop, average='macro'),
                'recall_macro': recall_score(y_val, y_pred_loop, average='macro'),
                'precision_macro': precision_score(y_val, y_pred_loop, average='macro'),
                'f1_class_0_early': f1_per_class[0],
                'f1_class_1_late': f1_per_class[1],
                'f1_class_2_healthy': f1_per_class[2],
                'seed': 42 + i,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }
            
            results.append(result)
            config_results.append(result)
            
        except Exception as e:
            log_message(f"   ⚠️  FEHLER in Run {i}: {e}")
            continue
    
    # Config abgeschlossen - Statistik und Checkpoint
    config_elapsed = time.time() - config_start_time
    
    if config_results:
        df_config = pd.DataFrame(config_results)
        log_message(f"\n  ✅ Config '{config['name']}' abgeschlossen:")
        log_message(f"     F1 Macro:  {df_config['f1_macro'].mean():.4f} ± {df_config['f1_macro'].std():.4f}")
        log_message(f"     Accuracy:  {df_config['accuracy'].mean():.4f} ± {df_config['accuracy'].std():.4f}")
        log_message(f"     Zeit:      {config_elapsed/60:.1f} Minuten")
        
        # Checkpoint speichern
        save_checkpoint(results, config_idx)
    else:
        log_message(f"  ❌ Config '{config['name']}' fehlgeschlagen!")

# 6. Finale Ergebnisse speichern
total_elapsed = time.time() - start_time_total

log_message(f"\n{'='*80}")
log_message(f"OVERNIGHT RUN ABGESCHLOSSEN")
log_message(f"Gesamtzeit: {total_elapsed/3600:.2f} Stunden")
log_message(f"{'='*80}")

if results:
    df_results = pd.DataFrame(results)
    final_file = RESULTS_DIR / f"overnight_robustness_{timestamp}.csv"
    df_results.to_csv(final_file, index=False)
    
    log_message(f"\n✅ Finale Datei gespeichert: {final_file}")
    log_message(f"   Anzahl Runs: {len(df_results)}")
    log_message(f"   Anzahl Configs: {df_results['config_name'].nunique()}")
    
    # Finale Zusammenfassung
    log_message(f"\n{'='*80}")
    log_message("FINALE ZUSAMMENFASSUNG PRO CONFIG")
    log_message(f"{'='*80}\n")
    
    for config_name in df_results['config_name'].unique():
        subset = df_results[df_results['config_name'] == config_name]
        log_message(f"📊 {config_name}:")
        log_message(f"   F1 Macro:       {subset['f1_macro'].mean():.4f} ± {subset['f1_macro'].std():.4f}")
        log_message(f"   Accuracy:       {subset['accuracy'].mean():.4f} ± {subset['accuracy'].std():.4f}")
        log_message(f"   ROC-AUC Macro:  {subset['roc_auc_macro'].mean():.4f} ± {subset['roc_auc_macro'].std():.4f}")
        log_message(f"   F1 Early:       {subset['f1_class_0_early'].mean():.4f} ± {subset['f1_class_0_early'].std():.4f}")
        log_message(f"   F1 Late:        {subset['f1_class_1_late'].mean():.4f} ± {subset['f1_class_1_late'].std():.4f}")
        log_message(f"   F1 Healthy:     {subset['f1_class_2_healthy'].mean():.4f} ± {subset['f1_class_2_healthy'].std():.4f}")
        log_message("")
    
else:
    log_message("\n❌ FEHLER: Keine Ergebnisse vorhanden!")

log_message(f"\nLogfile: {log_file}")

OVERNIGHT ROBUSTNESS RUN GESTARTET
Timestamp: 2026-03-01_22-41-29
Loops: 100

Anzahl Configs: 8
Geschätzte Gesamtzeit: 200.0 Minuten
(bei ~15 Sekunden pro Loop)


[1/8] Config: small_samples
  N_ensemble=32, n_samples=150


[1/8] small_samples:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
df_results

# 6.1 Robustness Vizualiszation

In [ ]:
# 4. Visualisierung der Robustness (Master-Thesis Style)
# Ziel: Durchschnittliche Performance zeigen + Stabilität (Fehlerbalken) beweisen

# Melt transformiert die Daten für Plotly (Wide -> Long Format)
import plotly.express as px
import os

df_melt = df_results.melt(
    id_vars=['run_id'], 
    value_vars=['f1_class_0_early', 'f1_class_1_late', 'f1_class_2_healthy'],
    var_name='Target Class', 
    value_name='F1 Score'
)

# Aggegierte Statistiken für Bar-Chart
df_stats = df_melt.groupby('Target Class')['F1 Score'].agg(['mean', 'std']).reset_index()

# Sortierung manuell festlegen (Logische Zeitreihe)
category_order = ['Früher Ausbruch (<1J)', 'Später Ausbruch (1-3J)', 'Kein Event / Gesund']
# Wir müssen die Werte in df_stats umbenennen, damit sie matchen, oder category_order anpassen
# Aktuell heißen die Werte 'f1_class_0_early' usw.
# Mapping Dictionary
name_mapping = {
    'f1_class_0_early': 'Früher Ausbruch (<1J)',
    'f1_class_1_late': 'Später Ausbruch (1-3J)',
    'f1_class_2_healthy': 'Kein Event / Gesund'
}
df_stats['Target Class'] = df_stats['Target Class'].map(name_mapping)
df_melt['Target Class'] = df_melt['Target Class'].map(name_mapping)

# 1. Bar Chart mit Error Bars (Klassisch wissenschaftlich)
fig = px.bar(
    df_stats, 
    x="Target Class", 
    y="mean", 
    error_y="std", # Zeigt die Standardabweichung als Antenne (Robustheit)
    title=f"Modell-Performance: Durchschnitt & Stabilität ({N_LOOPS} Runs)",
    text_auto='.1%', # Beschriftung direkt am Balken
    labels={'mean': 'Durchschnittlicher F1-Score'},
    color="Target Class", 
    # Pastell Farben wirken professioneller und weniger überladen
    color_discrete_sequence=px.colors.qualitative.Pastel,
    template="plotly_white",
    category_orders={"Target Class": category_order} # Erzwingt die logische zeitliche Reihenfolge
)

# Balken leicht transparent machen (0.8), damit sie nicht zu massiv wirken
fig.update_traces(marker_opacity=0.8, showlegend=False)

# 2. Scatter-Punkte darüber legen (Kontrastfarbe für bessere Sichtbarkeit)
# Zeigt jeden einzelnen Run als Punkt -> Maximale Transparenz der Ergebnisse
scatter_trace = px.strip(
    df_melt, 
    x="Target Class", 
    y="F1 Score", 
    category_orders={"Target Class": category_order}
).data[0]

# Styling der Punkte: Dunkles Kontrast-Blau mit weißem Rand
# Das sorgt für Lesbarkeit sowohl auf hellen als auch dunklen Hintergründen
scatter_trace.marker.color = '#34495e' # "Wet Asphalt" (Dunkles Grau-Blau)
scatter_trace.marker.size = 6
scatter_trace.marker.opacity = 0.8
scatter_trace.marker.line = dict(width=1, color='white') # Weißer Rand lässt Punkte "poppen"
scatter_trace.showlegend = False

fig.add_trace(scatter_trace)

# Achsen formatieren (Prozent statt 0.x)
fig.update_layout(
    yaxis_tickformat='.0%', 
    yaxis_title="F1 Score (Macro)", 
    xaxis_title=None, # X-Achsen Titel ist redundant wegen den Labels
    font=dict(size=14) # Schrift etwas größer für Thesis
)
fig.update_yaxes(range=[0, 1.1]) # Platz für Error Bars lassen

save_path = os.path.join(RESULTS_DIR, "robustness_scientific_bar.png")
fig.write_image(save_path)
print(f"Plot gespeichert unter: {save_path}")
fig.show()

Plot gespeichert unter: reports/robustness_experiment/TabPFN_v3/robustness_scientific_bar.png


In [ ]:
# 4b. Visualisierung der Feature Importance Stabilität
import pandas as pd
import plotly.express as px
import os

# Melt für Feature Wichtigkeit
feature_imp_cols = [c for c in df_results.columns if c.startswith("imp_")]

if len(feature_imp_cols) == 0:
    print("Keine Feature Importance Daten gefunden.")
    print("Der effiziente Loop oben (Zelle 32) berechnet nur Metriken für Speed.")
    print("Um diesen Plot zu sehen: Aktiviere Permutation Importance im Loop (dauert aber lange!)")
else: 
    df_melt_imp = df_results.melt(
        id_vars=['run_id'], 
        value_vars=feature_imp_cols,
        var_name='Feature', 
        value_name='Importance Score'
    )

    # Prefix "imp_" für schönere Labels entfernen
    df_melt_imp['Feature'] = df_melt_imp['Feature'].str.replace('imp_', '')

    # Sortierung berechnen: Wir wollen das wichtigste Feature OBEN haben + Median Sortierung
    sorted_features = df_melt_imp.groupby('Feature')['Importance Score'].median().sort_values(ascending=True).index.tolist()

    fig = px.box(
        df_melt_imp, 
        x="Importance Score", 
        y="Feature", 
        color="Feature", 
        # Wir erzwingen die Sortierung explizit über category_orders
        category_orders={"Feature": sorted_features}, 
        title=f"Feature Wichtigkeit über {N_LOOPS} Runs (Stabilitätstest)",
        points="all", 
        template="plotly_white",
        height=600,
        color_discrete_sequence=px.colors.qualitative.Pastel
    )

    fig.update_layout(showlegend=False) # Legend redundant if y-axis labeled

    # Punkte styling 
    fig.update_traces(marker=dict(opacity=0.6, size=3, color='#34495e')) 

    # Achsen formatieren
    fig.update_layout(
        xaxis_tickformat='.1%', 
        xaxis_title="Wichtigkeit (Mean Decrease Accuracy)",
        yaxis_title=None,
        font=dict(size=12)
    )

    # Speichern
    imp_save_path = os.path.join(RESULTS_DIR, "robustness_feature_importance.png")
    fig.write_image(imp_save_path)
    print(f"Plot gespeichert unter: {imp_save_path}")
    fig.show()

Keine Feature Importance Daten gefunden.
Der effiziente Loop oben (Zelle 32) berechnet nur Metriken für Speed.
Um diesen Plot zu sehen: Aktiviere Permutation Importance im Loop (dauert aber lange!)


# 6. Test

In [ ]:
# no fit, only predict on test set (1000 samples for speed)
X_test_sample = X_test.iloc[:1000]
y_test_sample = y_test[:1000]

print("Starte Test...")
y_test_pred = classifier.predict(X_test_sample)
print("Fertig!")

Starte Test...
Fertig!


In [ ]:
y_proba_test = classifier.predict_proba(X_test_sample) if sample else classifier.predict_proba(X_test)
# multi_class='ovr' berechnet den Durchschnitt der AUCs aller Klassen
roc_auc = roc_auc_score(y_test_sample if sample else y_test, y_proba_test, multi_class='ovr', average='macro')

# print stats
print(f"Accuracy: {accuracy_score(y_test_sample, y_test_pred):.2f}")
print(f"AUC - ROC Score: {roc_auc:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_sample, y_test_pred, 
                            target_names=['Früh (<1J)', 'Spät (>1J)', 'Gesund']))

Accuracy: 0.56
AUC - ROC Score: 0.82

Classification Report:
              precision    recall  f1-score   support

  Früh (<1J)       0.14      0.67      0.23        52
  Spät (>1J)       0.14      0.77      0.23        44
      Gesund       0.97      0.54      0.70       904

    accuracy                           0.56      1000
   macro avg       0.42      0.66      0.39      1000
weighted avg       0.89      0.56      0.65      1000

